## Hampshire Population

This notebook provides a lightweight estimate of Domesday Hampshire population density for comparative purposes. The goal is not to produce a new demographic reconstruction of Hampshire, but rather to derive a reasonable uncertainty distribution for population density that can be compared with corresponding estimates for Angkor. The estimate will account for uncertainty in the historical estimates of household size, which will be propagated into the population estimate for the county and then into the final density.

### Rationale

The underlying Domesday evidence is incomplete and population estimates have to be inferred/reconstructed. Household counts are recorded through social and fiscal categories rather than as a direct census, and converting these counts into population requires assumptions about persons per household. In addition, density estimation requires an area denominator, and even that is not entirely fixed because Hampshire's historical extent can vary slightly depending on the cartographic or administrative source used. Since the comparison in the paper concerns the broader Hampshire settlement system, the analysis here treats total Hampshire population over total Hampshire area, including the Isle of Wight.

Accordingly, this notebook models Hampshire density as a derived random quantity:

$$
\rho_H = \frac{P_H}{A_H}
$$

where:

- $P_H$ is total Hampshire population in 1086,
- $A_H$ is Hampshire land area under the historical boundary definition used here,
- $\rho_H$ is population density in persons per km$^2$.

### Population model

Population is estimated from Domesday household counts using a mean household size parameter:

$$
P_H = N_H \times s
$$

where:

- $N_H$ is the relevant Domesday household count or count-like population unit,
- $s$ is the mean number of persons per household.

The key uncertainty lies in $s$. Following La Poutré (2023), the limited medieval English evidence suggests household sizes around 5.3, 5.8, and 6.3 persons per household in later medieval contexts, but La Poutré argues that Domesday household size was likely smaller, highlighting Darby's suggested range of roughly 4--5 persons per household for 1086. This notebook therefore treats household size probabilistically, centering the prior in a Domesday-appropriate range while allowing some uncertainty around it informed by the variability in estimates reported by La Poutré.

La Poutré reviews the limited quantitative evidence available for household size in medieval England and discusses several reconstructed estimates derived from manorial demographic records.

| Estimate (persons/household) | Source                | Context                                                                                                                                                                    | Page  |
| ---------------------------- | --------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----- |
| ~5.3                         | Hallam (1958)         | Derived from manorial demographic data after correcting Hallam’s family-size figures to include all offspring and adjusting for widows/widowers                            | p. 92 |
| ~6.3                         | Howell (1983)         | Estimated from a 1280 tithing list for Kibworth Harcourt that records adult males; the population is reconstructed by doubling for females and adding children (<12 years) | p. 92 |
| ~5.8                         | La Poutré calculation | Derived from Glastonbury Abbey estates by combining customary tenants and landless sons and dividing estimated population by holdings                                      | p. 93 |

La Poutré notes that these estimates cluster around a mean of roughly 5.8 persons per household for the late-thirteenth-century medieval demographic peak**. However, he argues that household size in 1086 (Domesday) was likely smaller, because the late-medieval constraints on marriage and land access that inflated household size had not yet intensified. He therefore considers values below the thirteenth-century peak plausible and highlights Darby’s suggested Domesday range of roughly 4–5 persons per household.

### Area model

The area denominator is treated as fixed in the simplest version of the analysis, but the notebook is structured so that it can also be treated as uncertain if needed. This matters because historical Hampshire boundaries can differ slightly across modern GIS layers, historical county maps, and other reconstructed sources.

The area figures for Hampshire Ancient County derive from two 19th-century parliamentary census publications. The 1831 figure (1,018,550 acres) comes from the Enumeration Abstract (1833), a return compiled under an Act of the eleventh year of George IV's reign [1]; the 1851 figure (1,077,164 acres) comes from the Tables of Population and Houses (1851), which covered the divisions, registration counties, and districts of England and Wales [2]. Both figures were accessed via the GB Historical GIS / University of Portsmouth dataset (Vision of Britain) [3].

| Year | Acres | km² | Source |
|------|-------|-----|--------|
| 1831 | 1,018,550 | 4,122 | *Enumeration Abstract*, BPP 1833 xxxvi–xxxviii, Command No. 149 [1] |
| 1851 | 1,077,164 | 4,359 | *Tables of Population and Houses*, BPP 1851 xliii, 73, Command No. 1339 [2] |

Bibliography
[1] Great Britain. Abstracts of the Answers and Returns: Enumeration. Command No. 149. British Parliamentary Papers 1833 xxxvi–xxxviii. London: HMSO, 1833.

[2] Great Britain. Tables of Population and Houses in the Divisions, Registration Counties, and Districts of England and Wales. Command No. 1339. British Parliamentary Papers 1851 xliii, 73. London: HMSO, 1851.

[3] GB Historical GIS / University of Portsmouth. "Hampshire Ancient County through Time: Historical Statistics on Population, Area (acres)." A Vision of Britain through Time. Accessed March 9, 2026. https://www.visionofbritain.org.uk/unit/10204434/cube/AREA_ACRES. 

In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import pytensor.tensor as pt

In [2]:
# -----------------------------
# Data inputs
# -----------------------------

# Domesday Hampshire households from La Poutré (2023), chapter VIII
hampshire_rural_households = 10194
hampshire_urban_households = 1595
hampshire_total_households = hampshire_rural_households + hampshire_urban_households

# Hampshire land area in km^2, including the Isle of Wight
# Replace with your preferred historical estimate once confirmed.
hampshire_area_km2 = 4359

# Historically derived / reconstructed household-size means (persons per household)
# from La Poutré (2023)
household_size_estimates = np.array([5.3, 6.3, 5.8], dtype=float)

# Domesday-oriented reference range noted by La Poutré following Darby
darby_domesday_range = (4.0, 5.0)

print("Rural households:", hampshire_rural_households)
print("Urban households:", hampshire_urban_households)
print("Total households:", hampshire_total_households)
print("Hampshire area (km^2):", hampshire_area_km2)
print("Historical / reconstructed household-size means:", household_size_estimates)
print("Darby Domesday reference range:", darby_domesday_range)

Rural households: 10194
Urban households: 1595
Total households: 11789
Hampshire area (km^2): 4359
Historical / reconstructed household-size means: [5.3 6.3 5.8]
Darby Domesday reference range: (4.0, 5.0)


In [3]:
with pm.Model() as hampshire_density_model:
    # Fixed data
    n_households = pm.Data("n_households", hampshire_total_households)
    area_km2 = pm.Data("area_km2", hampshire_area_km2)
    hh_means_obs = pm.Data("hh_means_obs", household_size_estimates)

    # Domesday mean household size — prior centred on Darby's range
    mu_domesday = pm.Normal(
        "mu_domesday_household_size",
        mu=4.5,
        sigma=0.5,
    )

    # The three literature estimates are noisy observations of this mean
    sigma_recon = pm.Exponential("sigma_recon", lam=2)
    pm.Normal(
        "observed_means",
        mu=mu_domesday,
        sigma=sigma_recon,
        observed=hh_means_obs,
    )

    # Expected population
    lambda_population = pm.Deterministic(
        "lambda_population",
        n_households * mu_domesday,
    )

    # Total population count: Normal approximation to the sum of n_households
    # Poisson(mu_domesday) random variables. By CLT, sum ~ Normal(n*mu, sqrt(n*mu)).
    # Poisson variance is the minimum variance assumption consistent with a
    # counting distribution given only knowledge of the mean; no variance
    # information exists in the literature to justify a stronger assumption.
    total_population = pm.Normal(
        "total_population",
        mu=lambda_population,
        sigma=pt.sqrt(lambda_population),
    )

    density_per_km2 = pm.Deterministic(
        "density_per_km2",
        total_population / area_km2,
    )

    idata = pm.sample(
        draws=4000,
        tune=2000,
        chains=4,
        target_accept=0.95,
        random_seed=42,
    )
    
# Summary table
az.summary(
    idata,
    var_names=[
        "mu_domesday_household_size",
        "sigma_recon",
        "lambda_population",
        "total_population",
        "density_per_km2",
    ],
    hdi_prob=0.95,
)

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_domesday_household_size, sigma_recon, total_population]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 484 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu_domesday_household_size,5.279,0.406,4.446,6.010,0.008,0.006,2741.0,3213.0,1.0
sigma_recon,0.777,0.398,0.217,1.546,0.007,0.005,3265.0,4398.0,1.0
lambda_population,62237.022,4786.986,52415.710,70854.190,93.811,66.342,2741.0,3213.0,1.0
total_population,62234.911,4792.176,52335.154,70767.501,93.985,66.465,2738.0,3292.0,1.0
density_per_km2,14.277,1.099,12.006,16.235,0.022,0.015,2738.0,3292.0,1.0
